In [2]:
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [3]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download= True,
    transform= ToTensor()
)

test_data = datasets.FashionMNIST(
    root= "data",
    train= False,
    download= True,
    transform=ToTensor()
)



100%|██████████| 26.4M/26.4M [00:00<00:00, 41.1MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 793kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 16.2MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 17.4MB/s]


In [4]:
batch_size = 64

train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data,batch_size=batch_size)

for X,y in test_dataloader:
    print(f"Shape of X [N,C,H,W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N,C,H,W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [5]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [6]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [7]:
loss_fn = nn.CrossEntropyLoss()
optimiser = torch.optim.SGD(model.parameters(), lr = 1e-3)


In [8]:
def train(dataloader, model, loss_fn, optimiser):
    size = len(dataloader.dataset)
    model.train()
    for batch,  (X,y) in enumerate(dataloader):

        pred = model(X)
        loss = loss_fn(pred,y)

        loss.backward()
        optimiser.step()
        optimiser.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch+1)*len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [9]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss,  correct = 0,0
    with torch.no_grad():
        for X,y in dataloader:
            X,y = X.to(device), y.to(device)
            pred =  model(X)
            test_loss += loss_fn(pred,y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


In [10]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimiser)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.292579  [   64/60000]
loss: 2.277270  [ 6464/60000]
loss: 2.259865  [12864/60000]
loss: 2.266637  [19264/60000]
loss: 2.241260  [25664/60000]
loss: 2.208729  [32064/60000]
loss: 2.224766  [38464/60000]
loss: 2.180354  [44864/60000]
loss: 2.170625  [51264/60000]
loss: 2.152384  [57664/60000]
Test Error: 
 Accuracy: 37.4%, Avg loss: 2.137683 

Epoch 2
-------------------------------
loss: 2.140849  [   64/60000]
loss: 2.130923  [ 6464/60000]
loss: 2.067020  [12864/60000]
loss: 2.099237  [19264/60000]
loss: 2.039838  [25664/60000]
loss: 1.977143  [32064/60000]
loss: 2.010188  [38464/60000]
loss: 1.919409  [44864/60000]
loss: 1.916263  [51264/60000]
loss: 1.863613  [57664/60000]
Test Error: 
 Accuracy: 52.8%, Avg loss: 1.847102 

Epoch 3
-------------------------------
loss: 1.874696  [   64/60000]
loss: 1.847692  [ 6464/60000]
loss: 1.716184  [12864/60000]
loss: 1.777287  [19264/60000]
loss: 1.664967  [25664/60000]
loss: 1.618391  [32064/600